# Light curves

Pass observation times and microlensing parameters to obtain magnification.
Separations and source radii are in Einstein-radius units; times are in days.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
import matplotlib.pyplot as plt
from example._common import style
style()


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import singlens as sl


## Annual parallax

Set the sky position and reference epoch with `AnnualParallax`. Here times
are HJD − 2450000; keep `t_ref` fixed while fitting `t0` and the parallax components.
Use `time_origin=0` for absolute HJD, or `use_hjd=False` for JD TDB.


In [ ]:
times = 8000. + jnp.linspace(-40., 40., 1001)
geometry = sl.AnnualParallax(ra_deg=266., dec_deg=-29., t_ref=8000.)
event = dict(t0=8000., u0=.05, t_e=80., pi_e_n=.2, pi_e_e=-.1)
A_point = sl.magnification(times, **event, parallax=geometry)
A_uniform = sl.magnification(times, **event, rho=.1, parallax=geometry)
A_linear = sl.magnification(times, **event, rho=.1, a1=.5, parallax=geometry)


In [ ]:
from example.lightcurve.code.lightcurve import lightcurve_figure
fig = lightcurve_figure()
plt.show()


Omit `rho` for a point source, supply `rho` for a uniform disk, and add
physical `a1` for linear limb darkening. For a static trajectory, omit
`parallax`, `pi_e_n`, and `pi_e_e`.

## Repeated evaluation and flux

Use `Model` for repeated evaluation with named event parameters. Source and
blend fluxes give $F=f_s A+f_b$.


In [ ]:
model = sl.Model(source="linear", parallax=geometry)
params = sl.Parameters(**event, rho=.1, a1=.5)
evaluate = jax.jit(model.magnification)
A = evaluate(times, params)
flux = model.flux(times, params, f_s=1.2, f_b=.15)


Direct real separation or complex source position is also supported:


In [ ]:
A_at_position = model.magnification_at(.1 + .02j, rho=.1, a1=.5)
print(float(A_at_position))


Continue with [derivatives](autodiff.ipynb), [inference](inference.ipynb),
or the [API](../api.md). See [numerics](../numerics.md) for accuracy limits.
